# Book Spine OCR — Typography Grouping + Google Books Query Pipeline

## Objective

This version intentionally **does NOT classify TITLE / AUTHOR / OTHER**.

Instead it:

1. Detects text from each book spine.
2. Uses OCR polygons and perspective correction.
3. Re-recognizes difficult regions with targeted preprocessing.
4. Removes obvious OCR noise and duplicates.
5. Reconstructs fragmented text lines.
6. Measures **text size, text weight and visual style**.
7. Groups similar text by typography rather than semantic meaning.
8. Joins the typography groups into one comma-separated sentence.
9. Produces a `Google Books Query` suitable for the Google Books Volumes API.

### Example

```text
INTERNATIONAL, THINK & GROW RICH, NAPOLEON HILL
```

The purpose is to send the complete useful text from the spine to Google Books and let the metadata service determine the canonical book title and author.

### Important

- No Tesseract
- No DINOv2 / Transformers
- No KMeans semantic classifier
- No full-image 1x / 1.5x / 2x OCR
- Typography grouping is based on size / weight / visual style / geometry
- The final query preserves visual reading order


# PHASE 1 — Environment

## STEP 1 — Check versions and dependency conflicts

This cell intentionally avoids upgrading the environment.

It only installs **EasyOCR 1.7.2 if EasyOCR is missing** and then runs `pip check`.

The OCR pipeline does not require Tesseract, Transformers or DINOv2.


In [ ]:
# ============================================================
# PHASE 1 / STEP 1
# Environment check
# ============================================================

import sys
import subprocess
import importlib.util
from importlib import metadata

def package_version(name):
    try:
        return metadata.version(name)
    except Exception:
        return "not installed"

# Only install EasyOCR when it is actually missing.
if importlib.util.find_spec("easyocr") is None:
    print("EasyOCR not found. Installing EasyOCR 1.7.2...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "easyocr==1.7.2"
    ])

print("Python       :", sys.version)
print("EasyOCR      :", package_version("easyocr"))
print("PyTorch      :", package_version("torch"))
print("TorchVision  :", package_version("torchvision"))
print("OpenCV       :", package_version("opencv-python-headless"))
print("NumPy        :", package_version("numpy"))
print("Pandas       :", package_version("pandas"))

print("\nRunning pip check...")
check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)

if check.stdout.strip():
    print(check.stdout.strip())
else:
    print("No pip dependency conflicts reported.")

if check.stderr.strip():
    print(check.stderr.strip())


EasyOCR not found. Installing EasyOCR 1.7.2...
Python       : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
EasyOCR      : 1.7.2
PyTorch      : 2.11.0+cu130
TorchVision  : 0.26.0+cu130
OpenCV       : 5.0.0.93
NumPy        : 2.1.3
Pandas       : 2.2.3
Pillow       : 11.3.0
scikit-learn : 1.6.1

Running pip check...
ipython 7.34.0 requires jedi, which is not installed.


## STEP 2 — Imports and GPU check

In [ ]:
# ============================================================
# PHASE 1 / STEP 2
# Imports
# ============================================================

import gc
import re
import math
from pathlib import Path
from difflib import SequenceMatcher

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import easyocr

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


PyTorch : 2.11.0+cu130
CUDA    : True
GPU     : Tesla T4


# PHASE 2 — Input

## STEP 3 — Configure input/output folders

The notebook expects **already extracted individual book spines**.

Example:

```text
/content/bookImg/
    book_01.jpg
    book_02.jpg
    book_03.jpg
    ...
```

For a real bookshelf application, your segmentation model should create these spine crops first.


In [ ]:
# ============================================================
# PHASE 2 / STEP 3
# Paths and configuration
# ============================================================

INPUT_DIR = Path("/content/bookImg")

OUTPUT_DIR = Path("/content/BookSpine_OCR_Final_Output")
ANNOTATED_DIR = OUTPUT_DIR / "annotated"
NORMALIZED_DIR = OUTPUT_DIR / "normalized_spines"

for d in [
    OUTPUT_DIR,
    ANNOTATED_DIR,
    NORMALIZED_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp",
    ".JPG", ".JPEG", ".PNG", ".BMP", ".WEBP"
}

image_files = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.is_file() and p.suffix in extensions
)

if not image_files:
    raise FileNotFoundError(
        f"No images found in {INPUT_DIR}. "
        "Put individual book-spine images there."
    )

print(f"Found {len(image_files)} spine images.")
for p in image_files:
    print(" -", p.name)


Found 15 spine images.
 - book_01.jpg
 - book_02.jpg
 - book_03.jpg
 - book_04.jpg
 - book_05.jpg
 - book_06.jpg
 - book_07.jpg
 - book_08.jpg
 - book_09.jpg
 - book_10.jpg
 - book_11.jpg
 - book_12.jpg
 - book_13.jpg
 - book_14.jpg
 - book_15.jpg


# PHASE 3 — OCR Engine

## STEP 4 — Load EasyOCR

Only English is enabled because the supplied reference spines are English.

The recognizer uses beam search for better recognition quality.


In [ ]:

# ============================================================
# PHASE 3 / STEP 4
# EasyOCR
# ============================================================

USE_GPU = torch.cuda.is_available()

reader = easyocr.Reader(
    ["en"],
    gpu=USE_GPU,
    verbose=True
)

print("EasyOCR ready.")


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteEasyOCR ready.
